# Email Spam Classifier - AWS Marketplace

Deploys **Email Spam Classifier** from AWS Marketplace as a SageMaker endpoint inside **your own AWS account**. Your data stays in your account and there are no external API calls.

17.6MB spam classifier for email and messages on ml.t2.medium. Measured 98.6% accuracy on 2,000 test emails from the Enron-Spam dataset (98.5% precision, 98.7% recall), returns ham and spam probabilities. Apache-2.0, private in your VPC.

## Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

Cost: software fee $0.05/hr plus the SageMaker instance (ml.t2.medium real-time, ml.m5.large batch). Delete the endpoint when done.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.t2.medium"  # smallest real-time type, enough for this model
ENDPOINT_NAME = "bert-tiny-enron-spam-detection"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Creating the endpoint takes a few minutes. It bills per hour while it exists, so do not skip section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Classify text

The endpoint accepts `{"inputs": ["text", ...], "top_k": 1}` and returns `predictions`: one object per text with `label`, `score` and the full `scores` map.

In [ ]:
runtime = boto3.client("sagemaker-runtime")

PAYLOAD = {'inputs': ['Congratulations! You have been selected for a free prize.', "Agenda attached for tomorrow's meeting."], 'top_k': 1}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(PAYLOAD),
)
result = json.loads(response["Body"].read())
print(json.dumps(result)[:600])

for text, p in zip(PAYLOAD["inputs"], result["predictions"]):
    print(f"{p['label']:>6}  {p['score']:.3f}  {text}")

## 4. Batch transform for offline workloads

Batch transform avoids paying for an always-on endpoint (ml.m5.large). The input file is sent as ONE request, so put a single JSON request object (same shape as above) in the file.

In [ ]:
# transformer = model.transformer(
#     instance_count=1,
#     instance_type="ml.m5.large",
#     output_path=f"s3://{session.default_bucket()}/bert-tiny-enron-spam-detection/",
#     strategy="SingleRecord",
# )
# transformer.transform(
#     data=f"s3://{session.default_bucket()}/bert-tiny-enron-spam-detection-input/request.jsonl",
#     content_type="application/json",
# )
# transformer.wait()

## 5. Clean up

Delete the endpoint when you are done. It bills for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)